In [71]:
##Loading the dataset
import pandas as pd
data=pd.read_csv('all_kindle_review.csv.xls')
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [72]:
df=data[['reviewText','rating']]

In [73]:
df.sample(5)

,reviewText,rating
7852,Gage and Hailey are so sweet together I almost...,5
6185,Warning: This review might contain what some p...,4
6348,I got about two pages into this before giving ...,1
9537,The title caught my interest. Reading the fre...,1
5421,Will has been in love with his best friend sin...,3


In [74]:
df.shape

(12000, 2)

In [75]:
##Missing Values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [76]:
df.duplicated().sum()

0

In [77]:
df['rating'].unique()

array([3, 5, 4, 2, 1], dtype=int64)

In [78]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

### Preprocessing 

In [79]:
df=df.copy()

In [80]:
def convert_sentiment(rating):
    if rating <= 2:
        return 0   # negative
    elif rating >= 4:
        return 1   # positive
    else:
        return None

df['sentiment'] = df['rating'].apply(convert_sentiment)
df = df.dropna(subset=['sentiment'])

In [81]:
print(df['sentiment'].value_counts())

sentiment
1.0    6000
0.0    4000
Name: count, dtype: int64


In [82]:
df['reviewText'].str.lower()

1        great short read.  i didn't want to put it dow...
4        i did not expect this type of book to be in li...
5        aislinn is a little girl with big dreams. afte...
6        this has the makings of a good story... unfort...
7        i got this because i like collaborated short s...
                               ...                        
11992    i stumbled across this book and thought i woul...
11993    i really love the fine leather cases that are ...
11995    valentine cupid is a vampire- jena and ian ano...
11996    i have read all seven books in this series. ap...
11998    tried to use it to charge my kindle, it didn't...
Name: reviewText, Length: 10000, dtype: object

In [94]:
from nltk.corpus import stopwords

In [84]:
import re
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

def tokenize_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z0-9!? ]', '', text)
        return word_tokenize(text)
    return []

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\goyal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\goyal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [85]:
df['reviewText'].apply(tokenize_text)

1        [great, short, read, i, didnt, want, to, put, ...
4        [i, did, not, expect, this, type, of, book, to...
5        [aislinn, is, a, little, girl, with, big, drea...
6        [this, has, the, makings, of, a, good, story, ...
7        [i, got, this, because, i, like, collaborated,...
                               ...                        
11992    [i, stumbled, across, this, book, and, thought...
11993    [i, really, love, the, fine, leather, cases, t...
11995    [valentine, cupid, is, a, vampire, jena, and, ...
11996    [i, have, read, all, seven, books, in, this, s...
11998    [tried, to, use, it, to, charge, my, kindle, i...
Name: reviewText, Length: 10000, dtype: object

In [86]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english')) - {'not'}

def remove_stopword(text):
    tokens = text.split()
    filtered = [word for word in tokens if word not in stop_words]
    return " ".join(filtered)

In [87]:
df['reviewText'].apply(remove_stopword)

1        Great short read. I want put I read one sittin...
4        I not expect type book library pleased find pr...
5        Aislinn little girl big dreams. After death ol...
6        This makings good story... unfortunately disap...
7        I got I like collaborated short stories. Alot ...
                               ...                        
11992    I stumbled across book thought I would give tr...
11993    I really love fine leather cases available kin...
11995    Valentine cupid vampire- Jena Ian another vamp...
11996    I read seven books series. Apocalyptic/Adventu...
11998    tried use charge kindle, even register chargin...
Name: reviewText, Length: 10000, dtype: object

In [88]:
from nltk.corpus import wordnet

def lemmatize_words(text):
    if not isinstance(text, str):
        return ''
    
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if re.match(r'\w+', word)]
    
    return ' '.join([wordnet_lemmatizer.lemmatize(word, wordnet.VERB) for word in tokens])

In [89]:
df['cleaned_text']=df['reviewText'].apply(lemmatize_words)

In [90]:
df.head()

,reviewText,rating,sentiment,cleaned_text
1,Great short read. I didn't want to put it dow...,5,1.0,great short read i do n't want to put it down ...
4,I did not expect this type of book to be in li...,4,1.0,i do not expect this type of book to be in lib...
5,Aislinn is a little girl with big dreams. Afte...,5,1.0,aislinn be a little girl with big dream after ...
6,This has the makings of a good story... unfort...,2,0.0,this have the makings of a good story unfortun...
7,I got this because I like collaborated short s...,4,1.0,i get this because i like collaborate short st...


In [91]:
# TF-IDF
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=5
)

X = tfidf.fit_transform(df['cleaned_text'])
y = df['sentiment']

# Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB(alpha=0.5)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
from sklearn.metrics import accuracy_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8925
[[ 671  129]
 [  86 1114]]


In [93]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.89      0.84      0.86       800
         1.0       0.90      0.93      0.91      1200

    accuracy                           0.89      2000
   macro avg       0.89      0.88      0.89      2000
weighted avg       0.89      0.89      0.89      2000

